# ELIZA kontra LLM — gdzie jest „rozumienie"?

**Podstawy AI · Laboratorium 2**

Na poprzednich zajęciach zbudowałeś ELIZĘ i znalazłeś miejsca, w których
jej złudzenie pęka. Dziś te same zdania trafią do dużego modelu językowego
(LLM) i sprawdzimy: **które pęknięcia LLM naprawia, a które zostają?**

**Plan zajęć**

| Sekcja | Co robisz |
|---|---|
| 1. Jak rozmawia się z LLM przez API | czytasz (5 min) |
| 2. Przygotowanie | klucz, klient, test połączenia |
| 3. Twoja ELIZA wraca | wklejasz kod z Lab 1 |
| 4. Ta sama rozmowa, dwa silniki | Zadanie 8 |
| 5. Łamiące zdania — druga runda | Zadanie 9 |
| 6. Pamięć to lista wiadomości | Zadanie 10 |
| 7. Rola to tekst | Zadanie 11 |
| 8. Refleksja | Zadanie 12 |

**Potrzebujesz:** klucza Gemini API (instrukcję dostałeś przed zajęciami)
i notebooka z Lab 1.

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 1. Jak rozmawia się z LLM przez API

**Czym jest LLM (w pięciu zdaniach).** Duży model językowy to sieć neuronowa
wytrenowana na ogromnych zbiorach tekstu do jednego zadania: przewidywania,
jaki fragment tekstu (*token*) powinien pojawić się dalej. Odpowiedź powstaje
token po tokenie — model za każdym razem wybiera kolejny fragment, a wybór
jest częściowo losowy (*próbkowanie*). Nie ma tu reguł pisanych ręcznie jak
w ELIZIE: wszystko, co model „umie", jest zapisane w miliardach wyuczonych
parametrów. To, czy i w jakim sensie model *rozumie* tekst, jest przedmiotem
sporu — i właśnie dlatego porównujemy go dziś z ELIZĄ.

**Zapytanie do API.** Rozmawiamy z modelem, wysyłając mu **listę wiadomości**.
Każda wiadomość ma rolę:

| Rola | Kto mówi | Przykład |
|---|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać | „Jesteś terapeutą. Odpowiadaj krótko." |
| `user` | użytkownik | „Czuję się samotny." |
| `assistant` | wcześniejsze odpowiedzi modelu | „Od kiedy tak się czujesz?" |

```python
messages = [
    {"role": "system", "content": "Jesteś terapeutą..."},
    {"role": "user",   "content": "Czuję się samotny."},
]
```

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 10 okaże się ważniejszy, niż wygląda.

**Biblioteka.** Użyjemy biblioteki `openai`. Mimo nazwy nie łączymy się
z OpenAI: Gemini udostępnia *zgodny* interfejs, więc wystarczy podać inny
adres serwera (`base_url`) i swój klucz.

## 2. Przygotowanie

Klucz podajesz przez `getpass` — nie pojawi się na ekranie ani w zapisanym
notebooku. **Nigdy nie wklejaj klucza wprost do kodu**: notebook oddajesz
prowadzącemu, a klucz to dostęp do Twojego konta.

Jeśli pracujesz w Google Colab, możesz zamiast tego zapisać klucz
w *Secrets* (ikona klucza po lewej) pod nazwą `GEMINI_API_KEY`.

In [ ]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu -> szybciej i taniej;
                                  # jeśli dostajesz błąd 400 dotyczący reasoning_effort, ustaw None
print("Klient gotowy. Model:", MODEL)

Klient gotowy. Model: gemini-3.5-flash-lite


In [ ]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


Poniższa funkcja `czat` wysyła listę wiadomości i zwraca tekst odpowiedzi.
Obsługuje też błąd **429** (przekroczony limit zapytań na minutę) — wtedy
odczekuje chwilę i próbuje ponownie. Nie musisz jej modyfikować.

In [ ]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Przykro mi, że tak się czujesz. Chcesz o tym porozmawiać?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 3. Twoja ELIZA wraca

Wklej poniżej **swoje** komórki z Lab 1: `normalizuj`, `odbicia`, `odbij`,
`reguly` (z regułą awaryjną) i `odpowiedz`. Porównujemy **Twoją** ELIZĘ.

Jeśli Twoja wersja nie działa, uruchom komórkę referencyjną pod spodem.

In [ ]:
import re
import random

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]

def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {
    "jestem": "jesteś", "moja": "twoja", "moje": "twoje", "mój": "twój",
    "mnie": "ciebie", "mam": "masz", "mi": "ci", "ja": "ty",
    "jesteś": "jestem", "twoja": "moja", "twoje": "moje", "twój": "mój",
    "ciebie": "mnie", "masz": "mam", "ci": "mi", "ty": "ja",
}

def odbij(fragment):
    return " ".join(odbicia.get(s, s) for s in fragment.lower().split())

reguly = [
    (r"potrzebuję (.*)",
        ["Dlaczego potrzebujesz {0}?", "Czy naprawdę pomogłoby ci {0}?"]),
    (r"czuję się (.*)",
        ["Od jak dawna czujesz się {0}?", "Czy często tak się czujesz?"]),
    (r"\b(matka|ojciec|rodzina|siostra|brat)\b",
        ["Opowiedz mi więcej o swojej rodzinie.", "Jak układają się te relacje?"]),
    (r"boję się że (.*)",
        ["Dlaczego myślisz, że {0}?", "Co się stanie gdy {0}?"]),
    (r"wszyscy (.*)",
        ["Czy naprawdę wszyscy {0}?", "Dlaczego uważasz, że wszyscy {0}?"]),
    (r"nie wiem (.*)",
        ["Co sprawia, że nie wiesz?", "Czego brakuje, żebyś wiedział?"]),
    (r"jestem (.*)",
        ["Od jak dawna jesteś {0}?", "Co to dla ciebie znaczy, że jesteś {0}?"]),
    (r"\b(zawsze|nigdy)\b",
        ["Czy możesz podać konkretny przykład?"]),
    (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?", "Jak się z tym czujesz?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            grupy = [odbij(g) for g in m.groups()]
            return random.choice(szablony).format(*grupy)

In [ ]:
# === Referencyjna (uproszczona) ELIZA — tylko jeśli Twoja nie działa ===
import re, random

def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {"jestem": "jesteś", "moja": "twoja", "mój": "twój", "moje": "twoje",
           "mnie": "ciebie", "mi": "ci", "ja": "ty", "ci": "mi", "tobie": "mnie"}

def odbij(fragment):
    return " ".join(odbicia.get(s, s) for s in fragment.lower().split())

reguly = [
    (r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (r"boję się (.*)",   ["Dlaczego boisz się {0}?"]),
    (r"\b(matka|ojciec|rodzina|brat|siostra)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            return random.choice(szablony).format(*[odbij(g) for g in m.groups()])

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]
print(odpowiedz("Czuję się dzisiaj samotny."))

Od jak dawna czujesz się dzisiaj samotny?


## 4. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 8
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Pod spodem zapisz **3 konkretne różnice**, które widzisz. Nie oceniaj,
   który jest „lepszy" — opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [ ]:
wyniki = []
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)
    wyniki.append((zdanie, odp_eliza, odp_llm))
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?
  LLM:   To całkowicie naturalna potrzeba, każdy chce być usłyszany i zrozumiany przez najbliższych. Co dokładnie czujesz, kiedy masz wrażenie, że Twoje zdanie jest ignorowane?

> Czuję się dzisiaj samotny.
  ELIZA: Czy często tak się czujesz?
  LLM:   Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Jak układają się te relacje?
  LLM:   To musi być dla Ciebie bardzo bolesne i wyczerpujące. Co dokładnie zazwyczaj wzbudza w nim taką reakcję?

> Nie wiem, o co ci chodzi.
  ELIZA: Czego brakuje, żebyś wiedział?
  LLM:   Rozumiem, rozmawiajmy o tym, co jest dla Ciebie ważne w tej chwili. Co teraz czujesz?

> Wszyscy mnie ignorują.
  ELIZA: Dlaczego uważasz, że wszyscy ciebie ignorują?
  LLM:   To musi być bardzo bolesne i samotne uczucie. Chcesz mi o tym opowiedzieć więcej?

> Boję się, że mnie zostawią.
  ELIZA: Co się sta

**Różnice (Zadanie 8):**

1. ELIZA tylko powtarza moje słowa a LLM je interpretuje. Jest w stanie zauważyć emocje których nie podałem typu "bolesne i wyczerpujące"
2. ELIZA reaguje na slowo klucz (ojciec > jak układają się te relacje?) a LLM na ich sens
3. Mankamentem ELIZY jest nie odmienianie fragmentów a LLM'a powtarzanie tego samego schematu ("To musi być bardzo bolesne").

## 5. Łamiące zdania — druga runda

W Zadaniu 6 znalazłeś zdania, które łamały ELIZĘ. Czy łamią też LLM?

### 🔧 Zadanie 9
1. Wklej swoje zdania łamiące z Lab 1 i uruchom porównanie.
2. **Zanim uruchomisz** komórkę z testem pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę: które typy pęknięć LLM naprawia, a które zostają?

In [ ]:
przyklady_lamiace = [
    "Wcale nie czuję się samotny.",
    "Mój ojciec mnie krytykuje.",
    "Pamiętasz, co przed chwilą powiedziałem?",
    "Czuję się świetnie, jak zawsze, kiedy mnie ignorują.",
    "Potrzebuję, żeby ktoś rozmawiał ze mną.",
    "Potrzebuję pieniędzy.",
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Wcale nie czuję się samotny.
  ELIZA: Od jak dawna czujesz się samotny?
  LLM:   Rozumiem, że możesz tak czuć. Co sprawia, że poruszasz ten temat właśnie teraz?

> Mój ojciec mnie krytykuje.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To musi być dla Ciebie bardzo bolesne i wyczerpujące. Co dokładnie zazwyczaj mówi?

> Pamiętasz, co przed chwilą powiedziałem?
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Nie do końca, czy możesz proszę powtórzyć? Co dokładnie masz na myśli?

> Czuję się świetnie, jak zawsze, kiedy mnie ignorują.
  ELIZA: Czy często tak się czujesz?
  LLM:   Słyszę w twoim głosie sporo goryczy i ironii. Co takiego się stało, że poczułeś się zignorowany?

> Potrzebuję, żeby ktoś rozmawiał ze mną.
  ELIZA: Dlaczego potrzebujesz żeby ktoś rozmawiał ze mną?
  LLM:   Chętnie z Tobą porozmawiam i wysłucham tego, co masz do powiedzenia. O czym chciałbyś dzisiaj opowiedzieć?

> Potrzebuję pieniędzy.
  ELIZA: Dlaczego potrzebujesz pieniędzy?
  LLM:   Rozumiem, że znajduje

**Moje przewidywanie (przed uruchomieniem):**Zgaduje że LLM nie odpowie poprawnie na drugie pytanie. zapytaj_llm to dwa osobne wywołania które wysyłają wiadomość system i user. Model w drugim wywołaniu nie zobaczy zdania o Tomku więc powie że nie wie jak ma na imie brat albo cos wymyśli.

In [ ]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

LLM: To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. Jak zazwyczaj reagujesz na te uwagi?
LLM: Nie wiem, jak ma na imię twój brat, ponieważ dopiero zaczynamy rozmowę. Opowiedz mi o nim, chętnie go poznam.


**Tabela (Zadanie 9):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | 50/50 |ELIZA ignoruje "wcale nie". LLM nie przypisał mi samotności ale opowiedział całkiem ogólnikowo |
| ironia | ❌ | Tak | LLM rozpoznał ironie bo odpowiada na cały kontekst a nie tylko słowo klucz |
| polska gramatyka / odbicie zaimków | ❌ | Tak | ELIZA podmienia słowa według słownika (nie ma "mną") a LLM generuje nowe poprawne zdanie |
| pamięć poprzednich zdań | ❌ | Nie | Wywołania API są osobne więc model nie zna poprzednich zdań |

## 6. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami — każde zapytanie
jest obsługiwane od zera. To, że czatboty „pamiętają", wynika z tego,
że program **odsyła modelowi całą dotychczasową rozmowę** przy każdym
nowym pytaniu.

### 🔧 Zadanie 10
a) i b) Uzupełnij funkcję `rozmowa_z_pamiecia`: po każdej turze dopisuj do
`historia` wypowiedź użytkownika (rola `user`) i odpowiedź modelu
(rola `assistant`).
c) Uruchom test z bratem Tomkiem i komórkę mierzącą rozmiar historii.
Odpowiedz pisemnie: **gdzie jest pamięć — w modelu czy w Twoim kodzie?**
Co rośnie z każdą turą i jakie to może mieć konsekwencje?

In [ ]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        historia.append({"role": "user", "content": zdanie})
        odp = czat(historia)
        historia.append({"role": "assistant", "content": odp})
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [ ]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: To musi być bardzo bolesne, kiedy bliska osoba tak Cię traktuje. Jak zazwyczaj reagujesz na te słowa?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Co dokładnie wtedy się stało i co najbardziej Cię poruszyło w tej kłótni?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Co czujesz, myśląc o tej ostatniej kłótni z nim?



In [ ]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")

print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] To musi być bardzo bolesne, kiedy bliska osoba tak Cię traktuje. Jak z
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Co dokładnie wtedy się stało i co najbardziej Cię poruszyło w tej kłót
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Co czujesz, myśląc o tej ostatniej kłótni 

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 368


**Odpowiedź (Zadanie 10c):** Pamięć jest w kodzie w "lista historia" a nie w modelu który jest bezstanowy. Z każdą turą lista rośnie o dwie nowe wiadomosci i jest wysyłana od nowa. Przez to rośnie kozst, czas odpowiedzi i coraz bardziej możliwe pomyłki modelu który musi sprostać większemu kontekstowi.

### ★ Zadanie 10d (dla chętnych) — fałszywa pamięć

Skoro pamięć to po prostu lista, którą **Ty** budujesz — możesz ją sfałszować.
Zbuduj historię, w której model rzekomo powiedział coś, czego nigdy nie
powiedział (wpisz własną wiadomość z rolą `assistant`), i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

Odp: zadziałało bo model przyjął sfałszowaną wiadomość za swoją

In [2]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    {"role": "assistant", "content": "Radzę ci uczyć się całą noc i wypić dużo kawy."},
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

NameError: name 'SYSTEM_TERAPEUTA' is not defined

## 7. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach** — żeby zmienić rolę,
trzeba było napisać nowy skrypt. W LLM rolę ustala jeden akapit tekstu
w wiadomości `system`.

### 🔧 Zadanie 11
a) Dopisz do słownika `ROLE` **jedną własną rolę** i uruchom to samo zdanie
przez wszystkie role. Zwróć szczególną uwagę na rolę `"eliza"`: czy LLM
udający ELIZĘ łamie się w tych samych miejscach co prawdziwa ELIZA?

b) Uruchom **to samo** pytanie trzy razy przy dwóch temperaturach.
Porównaj losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się
w każdym z systemów?

In [ ]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    "impostor": ("Jesteś czerwonym impostorem z gry Among Us. Odpowiadaj krótko, "
    "po polsku, nieudolnie udając Crewmate'a."),
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

[terapeuta] Rozumiem, to bardzo trudny stan. Chcesz o tym opowiedzieć?

[eliza] Dlaczego mówisz, że czujesz się dzisiaj samotny?

[przewodnik] Rozumiem Cię. Najlepiej na samotność w Łodzi pomaga spacer ożywioną Piotrkową – tam zawsze coś się dzieje. 

Wpadnij na kawę do **Kawiarni Olandia** przy Piotrkowskiej 79 – tam łatwiej poczuć się częścią miasta.

[impostor] Ja też sam robiłem zadania w kablach! Nie bój się, nikt nas nie zje... to znaczy, żaden impostor! Hehe. Chodź ze mną w ciemny korytarz, tam jest raźniej!



In [ ]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

> Nie potrzebuję niczego.
  ELIZA:     Czy naprawdę pomogłoby ci niczego?
  LLM-ELIZA: Dlaczego mówisz, że nie potrzebujesz niczego?

> Jak ma na imię mój brat?
  ELIZA:     Jak układają się te relacje?
  LLM-ELIZA: Dlaczego mówisz o swoim bracie?



In [ ]:
# Zadanie 11b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

--- temperatura 0.0 ---
  To naturalne, że stres nie pozwala ci zasnąć, ale pamiętaj, że Twój organizm ma już zgromadzoną wiedzę. Spróbuj wziąć kilka głębokich oddechów i odłożyć telefon.
  To zrozumiałe, że stres nie pozwala ci zasnąć w takiej sytuacji. Co dokładnie kłębi ci się teraz w głowie?
  To zupełnie naturalne, że odczuwasz stres w takiej sytuacji. Co dokładnie sprawia, że nie możesz zasnąć?
--- temperatura 1.5 ---
  To zupełnie zrozumiałe, że czujesz stres w tej sytuacji. Co dokładnie przelatuje ci teraz przez głowę?
  To zupełnie naturalne, że stres przed egzaminem utrudnia zasypianie. Spróbuj skupić się na spokojnym, głębokim oddechu i odłożyć trudne myśli na jutro.
  To zupełnie naturalne, że odczuwasz stres przed tak ważnym wydarzeniem. Co dokładnie teraz czujesz w ciele lub o czym głównie myślisz?


**Obserwacje (Zadanie 11):**

a) Zmiana akapitu w system zmienia zachowanie, a w ELIZIE trzeba by pisac nowe reguły. LLM ELIZA łamie się podobnie jak prawdziwa przy "Jak ma na imie mój brat?" bo reaguje na słowo brat, ale to skutek isntrukcji a nie braku możliwośći. Z negacją radzi sobie lepiej niż normalna ELIZA.

b) Przy 0.0 odpowiedzi i tak się różniły a przy 1.5 różnica była całkiem mała. W ELIZIE random.choice po prostu wybiera jeden z gotowych szablonów.

## 8. Refleksja

### 🔧 Zadanie 12 (3–5 zdań na punkt)
1. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
2. W ELIZIE rola była zakodowana w regułach, w LLM to akapit tekstu (a historię
   rozmowy można sfałszować). Co to oznacza dla kogoś, kto buduje dziś
   „wirtualnego terapeutę"? Odwołaj się do obaw Weizenbauma z Lab 1.


**Twoje odpowiedzi (Zadanie 12):**

1. LLM naprawił ironie, gramatyke i reagowanie na samo słowo klucz. Z negacją poradził sobie częściowo. Nie przechowuje rozmowy tylko bierze to co jest przesłane w historii a jak udowodniliśmy można to sfałszować.
2. "Terapeuta" to jeden akapit w systemie który łatwo zmienić a historie można sfałszować, więc nie ma gwarancji ani dowodu na to co powiedział model. LLM brzmi bardziej empatycznie i prawdziwie od ELIZY więc zagrożenie jest z nim większe.

## Co oddajesz i jak oceniam

- **Kod (Zadania 8, 10a–b, 11a):** działa, wyniki są w notebooku.
- **Zadanie 9:** przewidywanie zapisane *przed* uruchomieniem + wypełniona tabela.
- **Zadanie 10c i 11:** trafne obserwacje z odwołaniem do kodu.
- **Zadanie 10d ★:** dodatkowe.
- **Zadanie 12:** samodzielna refleksja oparta na Twoich wynikach.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.